# 02 Singleton | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: jedna instancja
2. 🔨 Implementacja naiwna (`__new__`)
3. 🔒 Singleton bezpieczny watkowo
4. 🐍 Wzorzec Borg i modul jako Singleton
5. ⚠️ Singleton a testowalnosc

## 1. 🔹 Problem: jedna instancja

Singleton to wzorzec kreacyjny zapewniajacy, ze klasa ma dokladnie
jedna instancje i udostepniajacy do niej globalny punkt dostepu.

Kiedy potrzebujemy jednej instancji?
- Konfiguracja aplikacji - jedna centralna konfiguracja
- Logger - wspolny dziennik zdarzen
- Pula polaczen z baza danych - ograniczona liczba polaczen
- Cache - wspolna pamiec podreczna

> ⚠️ Problem bez Singletona: dwa obiekty Config stworzone w roznych
> miejscach moga miec rozne wartosci, co prowadzi do trudnych
> do znalezienia bledow.

In [ ]:
# Problem: dwie instancje Config - rozne obiekty
class ConfigBad:
    def __init__(self):
        self.debug = False
        self.db_url = 'sqlite:///default.db'

cfg1 = ConfigBad()
cfg2 = ConfigBad()
cfg1.debug = True

print(f'cfg1.debug = {cfg1.debug}')   # True
print(f'cfg2.debug = {cfg2.debug}')   # False - niezauwazona rozbieznosc!
print(f'cfg1 is cfg2: {cfg1 is cfg2}') # False - rozne obiekty
print(f'id(cfg1)={id(cfg1)}, id(cfg2)={id(cfg2)}')

---

### 🐍 Cwiczenia - problem jednej instancji

1. Stworz dwa obiekty klasy `list` i sprawdz czy `id()` sa rozne.
   Nastepnie stwórz dwa aliasy tej samej listy i sprawdz `id()`.
2. Napisz klase `Counter` z atrybutem `count = 0` i metoda
   `increment()`. Stworz dwa obiekty i policz ile wynosi laczny
   licznik po 3 + 2 inkrementacjach.
3. *(Trudniejsze)* Udowodnij problem: napisz funkcje `load_config()`
   tworzaca nowy obiekt `Config` z losowym `session_id`. Wywolaj
   ja dwa razy i sprawdz czy `session_id` sie rozni.

In [ ]:
# Cwiczenie 1: id() dla list
list1 = [1, 2, 3]
list2 = [1, 2, 3]
alias = list1

print(f'id(list1) == id(list2): {id(list1) == id(list2)}')
print(f'id(list1) == id(alias): {id(list1) == id(alias)}')

In [ ]:
# Cwiczenie 2: Counter bez Singletona
class Counter:
    def __init__(self): self.count = 0
    def increment(self) -> None: ...

c1 = Counter()
c2 = Counter()
for _ in range(3): c1.increment()
for _ in range(2): c2.increment()
print(f'c1.count = {c1.count}')   # 3
print(f'c2.count = {c2.count}')   # 2
print(f'Laczny: {c1.count + c2.count}')  # 5 - ale c1 i c2 nie wiedza o sobie

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: problem z Config
import random

class Config:
    def __init__(self):
        self.session_id = random.randint(1000, 9999)

def load_config() -> Config:
    return Config()

cfg_a = load_config()
cfg_b = load_config()
print(f'session_id a = {cfg_a.session_id}')
print(f'session_id b = {cfg_b.session_id}')
print(f'Identyczne: {cfg_a.session_id == cfg_b.session_id}')  # False - problem!

## 2. 🔹 Implementacja naiwna (`__new__`)

Python tworzy obiekty w dwoch krokach:
1. `__new__(cls)` - tworzy i zwraca nowy obiekt (alokacja pamieci)
2. `__init__(self)` - inicjalizuje obiekt (ustawienie atrybutow)

Kontrolujac `__new__`, mozemy zwrocic istniejaca instancje zamiast
tworzyc nowa. Przechowujemy referencje w atrybucie klasy `_instance`.

Przeplyw:
- pierwsze wywolanie: `_instance is None` -> tworzymy obiekt, zapisujemy
- kolejne wywolania: `_instance is not None` -> zwracamy ten sam obiekt

> 💡 `super().__new__(cls)` wywoluje `object.__new__(cls)` - domyslny
> mechanizm tworzenia nowego obiektu w Pythonie.

In [ ]:
class SingletonNaive:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            print('Tworzenie nowej instancji...')
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        # __init__ jest wywolywany przy kazdym new(), ale obiekt jest ten sam
        if not hasattr(self, '_initialized'):
            self._initialized = True
            self.value = 0

    def set_value(self, v: int) -> None:
        self.value = v

s1 = SingletonNaive()   # Tworzenie nowej instancji...
s2 = SingletonNaive()   # (brak komunikatu - zwraca istniejaca)
s3 = SingletonNaive()

print(f's1 is s2: {s1 is s2}')   # True
print(f's1 is s3: {s1 is s3}')   # True

s1.set_value(42)
print(f's2.value = {s2.value}')  # 42 - ten sam obiekt!

---

### 🐍 Cwiczenia - implementacja naiwna

1. Zaimplementuj `AppConfig` jako Singleton z atrybutami `debug = False`
   i `log_level = 'INFO'`. Sprawdz tozsamosc dwoch instancji.
2. Dodaj do Singletona licznik `call_count` zliczajacy ile razy
   wywolano `__new__`. Wywolaj konstruktor 5 razy i sprawdz licznik.
3. *(Trudniejsze)* Napisz Singleton `RequestLog` z lista `requests`.
   Metoda `add(url)` dodaje wpis. Sprawdz ze dwa obiekty
   `RequestLog` maja te sama liste po dodaniu wpisow z obu.

In [ ]:
# Cwiczenie 1: AppConfig Singleton
class AppConfig:
    _instance = None
    def __new__(cls):
        ...
    def __init__(self):
        ...

cfg1 = AppConfig()
cfg2 = AppConfig()
print(f'cfg1 is cfg2: {cfg1 is cfg2}')   # True
print(f'debug: {cfg1.debug}, log_level: {cfg1.log_level}')

In [ ]:
# Cwiczenie 2: licznik wywolan __new__
class CountedSingleton:
    _instance = None
    call_count = 0

    def __new__(cls):
        cls.call_count += 1
        ...

for _ in range(5):
    CountedSingleton()
print(f'Wywolania __new__: {CountedSingleton.call_count}')  # 5
print(f'Instancje sa identyczne: {CountedSingleton() is CountedSingleton()}') # True

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: RequestLog Singleton
class RequestLog:
    _instance = None

    def __new__(cls):
        ...
    def __init__(self):
        ...
    def add(self, url: str) -> None:
        ...

log1 = RequestLog()
log2 = RequestLog()
log1.add('/api/users')
log2.add('/api/orders')
print(f'log1.requests: {log1.requests}')  # ['/api/users', '/api/orders']
print(f'log2.requests: {log2.requests}')  # identyczna lista!

## 3. 🔹 Singleton bezpieczny watkowo

Implementacja naiwna ma blad przy wielowatkowosci: dwa watki moga
jednoczesnie wejsc do warunku `if _instance is None` i obydwa
stworzyc nowe obiekty.

Rozwiazanie: blokada `threading.Lock`. Stosujemy wzorzec
"podwojnego sprawdzania" (double-checked locking):
1. Sprawdz bez blokady (szybka sciezka)
2. Wejdz do blokady tylko jesli instancja nie istnieje
3. Sprawdz ponownie wewnatrz blokady (ktos moze ja byl stworzyc)

Dlaczego dwa sprawdzenia? Po zwolnieniu blokady przez watek A,
watek B wchodzi do blokady - bez drugiego sprawdzenia znow
stworzyłby nowy obiekt.

> 💡 W CPythonie GIL czescowo chroni przed wyścigami, ale nie
> gwarantuje atomowosci. Blokada jest zawsze bezpieczniejsza.

In [ ]:
import threading
import concurrent.futures

class ThreadSafeSingleton:
    _instance = None
    _lock = threading.Lock()

    def __new__(cls):
        if cls._instance is None:          # pierwsze sprawdzenie (bez blokady)
            with cls._lock:                # wejdz do sekcji krytycznej
                if cls._instance is None:  # drugie sprawdzenie (wewnatrz blokady)
                    cls._instance = super().__new__(cls)
        return cls._instance

    @classmethod
    def _reset(cls) -> None:               # pomocne w testach
        cls._instance = None

# Test wielowatkowy: 20 watkow tworzy instancje
instances = []
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    futures = [executor.submit(ThreadSafeSingleton) for _ in range(20)]
    instances = [f.result() for f in futures]

unique_ids = set(id(inst) for inst in instances)
print(f'Unikalnych instancji: {len(unique_ids)}')   # 1 - Singleton dziala!
print(f'Wszystkie identyczne: {all(inst is instances[0] for inst in instances)}')

---

### 🐍 Cwiczenia - bezpieczenstwo watkowe

1. Uruchom 10 watkow tworzacych `SingletonNaive` i sprawdz
   ile unikalnych `id()` zostalo stworzonych.
2. Uruchom 10 watkow tworzacych `ThreadSafeSingleton` i sprawdz
   ile unikalnych `id()` zostalo stworzonych.
3. *(Trudniejsze)* Zaimplementuj `SafeCounter` jako Singleton
   z metoda `increment()` bezpieczna watkowo (uzyj Lock do ochrony
   atrybutu `count`). Uruchom 100 watkow, kazdy inkrementujacy 10 razy.

In [ ]:
# Cwiczenie 1: test SingletonNaive wielowatkowo
import concurrent.futures

results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
    futures = [ex.submit(SingletonNaive) for _ in range(10)]
    results = [f.result() for f in futures]

unique = set(id(r) for r in results)
print(f'Unikalne instancje: {len(unique)}')

In [ ]:
# Cwiczenie 2: test ThreadSafeSingleton wielowatkowo
import concurrent.futures

results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
    futures = [ex.submit(ThreadSafeSingleton) for _ in range(10)]
    results = [f.result() for f in futures]

unique = set(id(r) for r in results)
print(f'Unikalne instancje: {len(unique)}')  # 1

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: SafeCounter
# hint: uzyj odrebnego _count_lock do ochrony count
import threading, concurrent.futures

class SafeCounter:
    _instance = None
    _instance_lock = threading.Lock()
    _count_lock = threading.Lock()

    def __new__(cls):
        ...
    def __init__(self):
        ...
    def increment(self) -> None:
        ...
    @property
    def count(self) -> int:
        ...

counter = SafeCounter()
SafeCounter._reset() if hasattr(SafeCounter, '_reset') else None

def worker():
    c = SafeCounter()
    for _ in range(10):
        c.increment()

with concurrent.futures.ThreadPoolExecutor(max_workers=100) as ex:
    list(ex.map(lambda _: worker(), range(100)))

print(f'Oczekiwane: 1000, wynik: {SafeCounter().count}')  # 1000

## 4. 🔹 Wzorzec Borg i modul jako Singleton

**Wzorzec Borg (monostate)**
Inne podejscie niz klasyczny Singleton: zamiast jedna instancja,
mozemy miec wiele instancji - ale wspoldzielacych ten sam stan
(`__dict__`). Wynik jest taki sam: wspolny stan dla wszystkich.

Implementacja: klasa ma wspolny slownik `_shared_state`. W `__init__`
przypasowujemy `self.__dict__ = self._shared_state`.

**Modul jako Singleton**
W Pythonie modul jest ladowany do pamieci tylko raz i wszystkie
importy wskazuja na ten sam obiekt. Zmienne i funkcje na poziomie
modulu sa naturalnym Singletonem - prostszym i bezpieczniejszym
niz klasa z `__new__`.

> 💡 "Modul jako Singleton" to idiom Pythona - preferowany dla
> konfiguracji aplikacji. Prosty, bezpieczny, wbudowany w jezyk.

In [ ]:
# Wzorzec Borg: wspoldzielony stan, rozne instancje
class BorgSingleton:
    _shared_state: dict = {}

    def __init__(self):
        self.__dict__ = self._shared_state  # wspoldzielony slownik

b1 = BorgSingleton()
b2 = BorgSingleton()
b1.value = 42
b1.theme = 'dark'

print(f'b2.value = {b2.value}')   # 42
print(f'b2.theme = {b2.theme}')   # dark
print(f'b1 is b2: {b1 is b2}')   # False - to rozne obiekty
print(f'id(b1) == id(b2): {id(b1) == id(b2)}')

# Modul jako Singleton - symulacja
class _AppConfig:               # prywatna klasa modulu
    def __init__(self):
        self.debug = False
        self.db_url = 'sqlite:///app.db'
        self.max_connections = 10

# Jedyna instancja na poziomie modulu
_config = _AppConfig()

def get(key: str):
    return getattr(_config, key)

def set_config(key: str, value) -> None:
    setattr(_config, key, value)

set_config('debug', True)
print(f'debug = {get("debug")}')   # True

---

### 🐍 Cwiczenia - Borg i modul

1. Zaimplementuj `SharedSettings` wzorcem Borg. Sprawdz ze dwa
   obiekty maja ten sam stan ale rozne `id()`.
2. Napisz klase `ThemeManager` wzorcem Borg z atrybutami
   `primary_color`, `font_size`, `dark_mode`. Zmien `dark_mode`
   przez jeden obiekt i sprawdz drugi.
3. *(Trudniejsze)* Napisz dekorator `@singleton` zamieniajacy
   dowolna klase w Singleton. Zastosuj do klas `A` i `B` - sprawdz
   ze sa niezalezne (Singleton A != Singleton B).

In [ ]:
# Cwiczenie 1: SharedSettings wzorcem Borg
class SharedSettings:
    _shared_state: dict = {}
    def __init__(self): ...

s1 = SharedSettings()
s2 = SharedSettings()
s1.lang = 'pl'
print(f's2.lang = {s2.lang}')          # pl
print(f's1 is s2: {s1 is s2}')         # False
print(f'id(s1) == id(s2): {id(s1) == id(s2)}')  # False

In [ ]:
# Cwiczenie 2: ThemeManager wzorcem Borg
class ThemeManager:
    _shared_state: dict = {}

    def __init__(self):
        ...
        # inicjalizuj tylko raz
        if 'primary_color' not in self._shared_state:
            self.primary_color = '#333'
            self.font_size = 14
            self.dark_mode = False

tm1 = ThemeManager()
tm2 = ThemeManager()
tm1.dark_mode = True
print(f'tm2.dark_mode = {tm2.dark_mode}')  # True

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: dekorator @singleton
def singleton(cls):
    # hint: uzyj slownika instances = {} w domknieciu
    ...

@singleton
class ServiceA:
    def __init__(self): self.name = 'A'

@singleton
class ServiceB:
    def __init__(self): self.name = 'B'

a1, a2 = ServiceA(), ServiceA()
b1, b2 = ServiceB(), ServiceB()
print(f'a1 is a2: {a1 is a2}')   # True
print(f'b1 is b2: {b1 is b2}')   # True
print(f'a1 is b1: {a1 is b1}')   # False - rozne Singletony!

## 5. 🔹 Singleton a testowalnosc

Singleton jest "antypattenem" w kontekscie testow: globalny stan
przenika miedzy testami i testy moga na siebie wplywac.

Glowne problemy:
- Stan z poprzedniego testu pozostaje -> testy zalezne od kolejnosci
- Nie mozna latwio podmienic Singletona na mock
- Izolacja testow wymaga recznego resetu

Sposoby zarzadzania Singletonem w testach:
1. Metoda klasy `_reset_instance()` - zeruje `_instance`
2. `unittest.mock.patch` - podmienia klase na czas testu
3. Dependency Injection - zamiast Singletona wstrzykuj obiekt

> ⚠️ Singleton jest bezposrednim odpowiednikiem zmiennej globalnej.
> Im mniej Singletonow, tym testowalniejszy kod.

In [ ]:
from unittest.mock import patch

class TestableSingleton:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self):
        if not hasattr(self, '_initialized'):
            self._initialized = True
            self.value = 0

    @classmethod
    def _reset_instance(cls) -> None:
        cls._instance = None

# Symulacja testu 1
print('--- Test 1 ---')
s1 = TestableSingleton()
s1.value = 10
print(f'value = {s1.value}')   # 10

# Reset miedzy testami
TestableSingleton._reset_instance()

# Symulacja testu 2 - czysta instancja
print('--- Test 2 ---')
s2 = TestableSingleton()
print(f'value = {s2.value}')   # 0 - swiezy stan
print(f's1 is s2: {s1 is s2}') # False - to juz inny obiekt

# Uzycie mock.patch
with patch('__main__.TestableSingleton') as MockSingleton:
    MockSingleton.return_value.value = 999
    result = TestableSingleton()
    print(f'Mock value: {result.value}')  # 999

---

### 🐍 Cwiczenia - testowalnosc

1. Napisz Singleton `Cache` z metoda `set(k, v)` i `get(k)`.
   Nastepnie napisz dwa "testy" (funkcje) izolowane przez
   `_reset_instance()` - sprawdz ze stany sie nie mieszaja.
2. Uzyj `unittest.mock.patch` do podmiany Singletona `AppConfig`
   w funkcji `start_app()` zwracajacej wartosc config['debug'].
3. *(Trudniejsze)* Przepisz `Cache` tak, zeby zamiast Singletona
   przyjmowal obiekt cache jako argument (Dependency Injection).
   Udowodnij ze dwa wywolania tej samej funkcji moga uzywac
   roznych cache bez global state.

In [ ]:
# Cwiczenie 1: izolacja testow przez _reset_instance()
class Cache:
    _instance = None
    def __new__(cls): ...
    def __init__(self): ...
    def set(self, k: str, v) -> None: ...
    def get(self, k: str): ...
    @classmethod
    def _reset_instance(cls): ...

def test_set_and_get():
    Cache._reset_instance()
    c = Cache()
    c.set('user', 'Alice')
    assert c.get('user') == 'Alice'
    print('test_set_and_get: OK')

def test_miss_returns_none():
    Cache._reset_instance()
    c = Cache()
    assert c.get('missing') is None
    print('test_miss_returns_none: OK')

test_set_and_get()
test_miss_returns_none()

In [ ]:
# Cwiczenie 2: mock.patch
from unittest.mock import patch

class AppConfig:
    _instance = None
    def __new__(cls): ...
    def __init__(self): ...
        
def start_app() -> bool:
    cfg = AppConfig()
    return cfg.debug

# Normalnie debug = False
AppConfig._reset_instance() if hasattr(AppConfig, '_reset_instance') else None
print(f'Normal: {start_app()}')  # False

# Z mockiem: debug = True
with patch('__main__.AppConfig') as MockConfig:
    MockConfig.return_value.debug = True
    print(f'Mocked: {start_app()}')  # True

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Dependency Injection zamiast Singletona
# hint: cache jako parametr funkcji, nie globalny stan
class SimpleCache:
    def __init__(self): self._data: dict = {}
    def set(self, k: str, v) -> None: self._data[k] = v
    def get(self, k: str): return self._data.get(k)

def get_user(user_id: int, cache: SimpleCache) -> dict:
    result = cache.get(str(user_id))
    if result is None:
        result = {'id': user_id, 'name': f'User{user_id}'}
        cache.set(str(user_id), result)
    return result

cache1 = SimpleCache()
cache2 = SimpleCache()  # oddzielny cache!

get_user(1, cache1)
print(f'cache1: {cache1.get("1")}')   # {'id': 1, 'name': 'User1'}
print(f'cache2: {cache2.get("1")}')   # None - odizolowany!